# 02 — Fama-MacBeth (1992): 한국 vs 미국

`data/`에 저장된 parquet을 로드해 팩터 구성 → FM 회귀 → 시각화를 수행한다.  
수집 단계를 재실행하려면 `01_data_fetch.ipynb`의 `REFRESH_DATA = True`로 변경할 것.

## 1. 라이브러리 & 파라미터

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import platform
from pathlib import Path
import json as _json

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── 한글 폰트 ────────────────────────────────────────────────────────────────
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    try:
        import matplotlib.font_manager as fm
        nanum = [f for f in fm.findSystemFonts() if 'Nanum' in f]
        if nanum: plt.rcParams['font.family'] = fm.FontProperties(fname=nanum[0]).get_name()
    except Exception:
        pass
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

# ── 파라미터 ──────────────────────────────────────────────────────────────────
DATE_START       = '2015-01-01'
DATE_END         = '2026-03-31'
BETA_WINDOW      = 60    # 베타 추정 롤링 윈도우 (개월)
BETA_MIN_PERIODS = 36    # 최소 유효 관측 수 (이하면 NaN)
FM_MIN_STOCKS    = 30    # 횡단면 OLS 최소 종목 수
NW_LAGS          = 6     # Newey-West HAC 래그
N_WORKERS        = 20    # 병렬 스레드 수

DATA_DIR = Path('data')
print(f'DATA_DIR = {DATA_DIR.resolve()}')


## 2. 공통 함수

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 공통 유틸리티 함수
# ════════════════════════════════════════════════════════════════════════════

def resample_monthly(price_series: "pd.DataFrame | pd.Series") -> "pd.Series":
    """일별 가격 시계열을 월말 기준으로 리샘플링.

    Parameters
    ----------
    price_series : 일별 가격 Series 또는 단일 열 DataFrame

    Returns
    -------
    pd.Series : 월말(해당 월의 마지막 날) 인덱스를 가진 월별 종가 시계열
    """
    s = price_series.squeeze()
    s = s.resample('ME').last().dropna()
    s.index = s.index.to_period('M').to_timestamp('M')
    return s


def rolling_beta(rtn_df: "pd.DataFrame", mkt_rtn: "pd.Series") -> "pd.DataFrame":
    """CAPM 롤링 베타를 완전 벡터화로 계산 (종목별 반복문 없음).

    베타 = Cov(r_i, r_m) / Var(r_m),  윈도우=BETA_WINDOW(60개월).
    cumsum 트릭으로 T×N 행렬 연산만 사용 → 종목 수에 무관하게 빠름.

    Parameters
    ----------
    rtn_df  : 종목별 월별 수익률 DataFrame (T행 × N열)
    mkt_rtn : 시장 월별 수익률 Series (T,)

    Returns
    -------
    pd.DataFrame : 종목별 롤링 베타 (T×N).
                   최소 관측 수(BETA_MIN_PERIODS) 미달 구간은 NaN.
    """
    mkt = mkt_rtn.reindex(rtn_df.index)
    W   = BETA_WINDOW
    MP  = BETA_MIN_PERIODS

    def rolling_cov_vec(x: "pd.DataFrame", y: "pd.Series") -> "pd.DataFrame":
        """x(T×N)와 y(T,)의 롤링 공분산. Cov = E[xy] - E[x]E[y].

        결측값은 0으로 채운 뒤 유효 개수 마스크로 보정한다.
        """
        xv = x.notna() & y.notna().values.reshape(-1, 1)
        n  = xv.astype(float)
        xf = x.fillna(0.0).values
        yf = y.fillna(0.0).values.reshape(-1, 1)

        def roll_sum(a: "np.ndarray") -> "np.ndarray":
            """길이-W 슬라이딩 합을 cumsum으로 O(T) 계산."""
            cs = a.cumsum(axis=0)
            out = cs.copy()
            out[W:] = cs[W:] - cs[:-W]
            out[:W] = cs[:W]
            return out

        n_roll = roll_sum(n)
        sx  = roll_sum(xf * xv.values)
        sy  = roll_sum(yf * xv.values)
        sxy = roll_sum(xf * yf * xv.values)
        with np.errstate(invalid='ignore', divide='ignore'):
            cov = (sxy / n_roll) - (sx / n_roll) * (sy / n_roll)
            cov[n_roll < MP] = np.nan
        return pd.DataFrame(cov, index=x.index, columns=x.columns)

    def rolling_var_vec(y: "pd.Series") -> "pd.Series":
        """y(T,)의 롤링 분산. Var = E[y²] - E[y]²."""
        yv = y.notna()
        n  = yv.astype(float)
        yf = y.fillna(0.0).values

        def roll_sum_1d(a: "np.ndarray") -> "np.ndarray":
            cs = a.cumsum()
            out = cs.copy()
            out[W:] = cs[W:] - cs[:-W]
            out[:W] = cs[:W]
            return out

        n_roll = roll_sum_1d(n)
        sy  = roll_sum_1d(yf * yv.values)
        sy2 = roll_sum_1d(yf**2 * yv.values)
        with np.errstate(invalid='ignore', divide='ignore'):
            var = (sy2 / n_roll) - (sy / n_roll)**2
            var[n_roll < MP] = np.nan
        return pd.Series(var, index=y.index)

    cov_mat = rolling_cov_vec(rtn_df, mkt)
    mkt_var = rolling_var_vec(mkt)
    beta    = cov_mat.div(mkt_var, axis=0)
    return beta.dropna(how='all')


def fm_step1(
    rtn_df: "pd.DataFrame",
    factor_dict: "dict[str, pd.DataFrame]",
    lag: int = 1,
) -> "pd.DataFrame":
    """Fama-MacBeth 1단계: 매월 횡단면 OLS로 gamma_t 추정.

    각 시점 t:  r_{i,t} = gamma_0 + sum_k(gamma_k * F_{k,i,t-lag}) + e_{i,t}

    팩터를 lag 개월 래깅해 look-ahead bias를 방지한다.
    FM_MIN_STOCKS 미만인 월은 횡단면 회귀 불가로 스킵.

    Parameters
    ----------
    rtn_df      : 종목별 월별 수익률 (T×N)
    factor_dict : {'팩터명': 팩터 DataFrame(T×N)}
    lag         : 팩터 래그 개월 수 (기본 1)

    Returns
    -------
    pd.DataFrame : 월별 추정 계수 (인덱스=날짜, 열=const·팩터명·n_stocks)
    """
    lagged   = {k: v.shift(lag) for k, v in factor_dict.items()}
    records  = []
    ols_fail = 0

    for date in rtn_df.index:
        rtn_t = rtn_df.loc[date].dropna()
        fac_t = {k: lagged[k].loc[date].dropna()
                 for k in lagged if date in lagged[k].index}
        if not fac_t:
            continue

        valid = set(rtn_t.index)
        for fv in fac_t.values():
            valid &= set(fv.index)
        valid = list(valid)

        if len(valid) < FM_MIN_STOCKS:
            continue

        y = rtn_t.loc[valid]
        X = sm.add_constant(pd.DataFrame({k: fac_t[k].loc[valid] for k in fac_t}))
        try:
            params = sm.OLS(y, X).fit().params
            rec = {'date': date, 'n_stocks': len(valid)}
            rec.update(params.to_dict())
            records.append(rec)
        except Exception as e:
            ols_fail += 1
            if ols_fail <= 3:
                print(f"  [fm_step1] OLS 실패 {date}: {e}")

    if ols_fail > 0:
        print(f"  [fm_step1] 총 OLS 실패: {ols_fail}회")

    return pd.DataFrame(records).set_index('date')


def fm_step2(gamma_df: "pd.DataFrame") -> "pd.DataFrame":
    """Fama-MacBeth 2단계: gamma_t 시계열 평균 + Newey-West t-통계량.

    gamma_t에 시계열 자기상관이 있으므로 단순 OLS SE 대신
    Newey-West HAC 표준오차를 사용한다 (maxlags=NW_LAGS).
    유의수준: * 10%, ** 5%, *** 1%.

    Parameters
    ----------
    gamma_df : fm_step1 반환값 — 열: const, 팩터명들, n_stocks

    Returns
    -------
    pd.DataFrame : 팩터별 {Mean_gamma, Std, NW_SE, t_stat, p_val, Sig, N_months}
                   인덱스 = 팩터명 (const 포함)
    """
    factor_cols = [c for c in gamma_df.columns if c != 'n_stocks']
    rows = []

    for col in factor_cols:
        s = gamma_df[col].dropna()
        n = len(s)
        if n < 12:
            continue

        mean_val = s.mean()
        try:
            nw     = sm.OLS(s, np.ones(n)).fit(cov_type='HAC', cov_kwds={'maxlags': NW_LAGS})
            se     = float(nw.bse.iloc[0])
            t_stat = float(nw.tvalues.iloc[0])
        except Exception as e:
            print(f"  [fm_step2] Newey-West 실패 ({col}), OLS SE로 대체: {e}")
            se     = s.std() / np.sqrt(n)
            t_stat = mean_val / se if se > 0 else np.nan

        p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=n - 1))
        sig   = ('***' if abs(t_stat) > 2.576 else
                 '**'  if abs(t_stat) > 1.960 else
                 '*'   if abs(t_stat) > 1.645 else '')
        rows.append({
            'Factor': col, 'Mean_gamma': mean_val, 'Std': s.std(),
            'NW_SE': se, 't_stat': t_stat, 'p_val': p_val,
            'Sig': sig, 'N_months': n,
        })

    return pd.DataFrame(rows).set_index('Factor')


def fetch_yf_fundamentals(ticker_yf: str) -> "dict[str, float]":
    """yfinance에서 단일 종목의 주식수와 주당장부가치를 수집.

    fast_info에서 주식수를 우선 시도하고, 없으면 full info로 폴백.
    네트워크 오류 발생 시 NaN 딕셔너리를 반환해 파이프라인 중단을 방지.

    Parameters
    ----------
    ticker_yf : yfinance 티커 심볼 (예: 'AAPL', 'BRK-B')

    Returns
    -------
    dict : {'ticker_yf': str, 'shares': float, 'book_per_share': float}
           오류 시 shares=NaN, book_per_share=NaN
    """
    try:
        info   = yf.Ticker(ticker_yf).fast_info
        shares = getattr(info, 'shares', None)
        if not shares:
            full   = yf.Ticker(ticker_yf).info
            shares = full.get('sharesOutstanding') or full.get('impliedSharesOutstanding')
            bv     = full.get('bookValue', np.nan)
        else:
            full = yf.Ticker(ticker_yf).info
            bv   = full.get('bookValue', np.nan)
        return {'ticker_yf': ticker_yf, 'shares': shares or np.nan, 'book_per_share': bv}
    except Exception as e:
        print(f"  [fetch_yf] {ticker_yf} 실패: {e}")
        return {'ticker_yf': ticker_yf, 'shares': np.nan, 'book_per_share': np.nan}


def parallel_fundamentals(
    tickers_yf: "list[str]",
    n_workers: int = N_WORKERS,
) -> "pd.DataFrame":
    """ThreadPoolExecutor로 yfinance 기본데이터를 병렬 수집.

    Parameters
    ----------
    tickers_yf : yfinance 티커 리스트
    n_workers  : 스레드 수 (yfinance rate limit 고려해 과도하게 늘리지 말 것)

    Returns
    -------
    pd.DataFrame : 인덱스=ticker_yf, 열={shares, book_per_share}
    """
    results = []
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(fetch_yf_fundamentals, t): t for t in tickers_yf}
        for f in tqdm(as_completed(futures), total=len(futures), desc='Fundamentals'):
            results.append(f.result())
    df = pd.DataFrame(results).set_index('ticker_yf')
    df['shares']         = pd.to_numeric(df['shares'],         errors='coerce')
    df['book_per_share'] = pd.to_numeric(df['book_per_share'], errors='coerce')
    return df


def run_fama_macbeth(
    rtn_df: "pd.DataFrame",
    beta_df: "pd.DataFrame",
    ln_me_df: "pd.DataFrame",
    ln_bm_df: "pd.DataFrame",
    label: str = '',
) -> "tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str]]":
    """Fama-MacBeth 3개 모델을 일괄 실행하는 편의 함수.

    Model 1: Beta only                    — CAPM 단독 설명력
    Model 2: Beta + ln(ME) + ln(B/M)      — FF(1992) 완전 모형
    Model 3: ln(ME) + ln(B/M) (Beta 제외) — 사이즈·가치 효과만

    Parameters
    ----------
    rtn_df   : 종목별 월별 수익률 (T×N)
    beta_df  : 롤링 베타 (T×N)
    ln_me_df : log 시가총액 (T×N)
    ln_bm_df : log 장부가치비율 (T×N)
    label    : 출력용 시장 이름 (예: '한국', '미국')

    Returns
    -------
    res_m1, res_m2, res_m3 : 각 모델의 fm_step2 결과 DataFrame
    gamma_m2               : Model 2 월별 gamma 시계열 (C-2 시각화용)
    common                 : 3개 팩터 공통 보유 종목 리스트
    """
    common = list(set(beta_df.columns) & set(ln_me_df.columns) & set(ln_bm_df.columns))
    rtn_fm = rtn_df[common]
    factor_dict = {
        'Beta':  beta_df[common],
        'ln_ME': ln_me_df[common],
        'ln_BM': ln_bm_df[common],
    }
    print(f'[{label}] FM 공통 종목: {len(common):,}개')

    gamma_m1 = fm_step1(rtn_fm, {'Beta': beta_df[common]})
    res_m1   = fm_step2(gamma_m1)

    gamma_m2 = fm_step1(rtn_fm, factor_dict)
    res_m2   = fm_step2(gamma_m2)

    gamma_m3 = fm_step1(rtn_fm, {'ln_ME': ln_me_df[common], 'ln_BM': ln_bm_df[common]})
    res_m3   = fm_step2(gamma_m3)

    print(f'=== {label} 결과 (Model 2) ===')
    print(res_m2[['Mean_gamma', 'NW_SE', 't_stat', 'Sig', 'N_months']].to_string())
    return res_m1, res_m2, res_m3, gamma_m2, common


print('FM 공통 함수 정의 완료')


## 3. 데이터 로드

In [ ]:
# ── 데이터 로드 ───────────────────────────────────────────────────────────────
# 01_data_fetch.ipynb에서 저장한 parquet/json을 로드.
# 이 셀만 성공하면 이후 셀은 인터넷 연결 없이 실행 가능.
kr_price   = pd.read_parquet(DATA_DIR / 'kr_price.parquet')
kr_mkt_rtn = pd.read_parquet(DATA_DIR / 'kr_mkt_rtn.parquet')['KR_market']
kr_listing  = pd.read_parquet(DATA_DIR / 'kr_listing.parquet')
book_df     = pd.read_parquet(DATA_DIR / 'kr_equity.parquet')
book_df.columns = [int(c) for c in book_df.columns]

us_price   = pd.read_parquet(DATA_DIR / 'us_price.parquet')
us_mkt_rtn = pd.read_parquet(DATA_DIR / 'us_mkt_rtn.parquet')['US_market']
valid_us   = pd.read_parquet(DATA_DIR / 'us_fund.parquet')
us_bv_annual = {k: {int(yr): v for yr, v in d.items()}
                for k, d in _json.loads((DATA_DIR / 'us_bv_annual.json').read_text(encoding='utf-8')).items()}

print('로드 완료')
print(f'  kr_price  : {kr_price.shape}  |  us_price  : {us_price.shape}')
print(f'  kr_mkt_rtn: {len(kr_mkt_rtn)}개월  |  us_mkt_rtn: {len(us_mkt_rtn)}개월')
print(f'  kr_listing: {len(kr_listing):,}종목  |  valid_us   : {len(valid_us):,}종목')
print(f'  book_df   : {book_df.shape} (한국 연도별 자본총계)')


---
## A. 한국

## A-2. 수익률 계산 (클리핑)

In [ ]:
# ── A-2: 한국 월별 수익률 계산 + 클리핑 ──────────────────────────────────────
# 클리핑 범위: -90% ~ +900% (상장폐지 직전 급락, 스팸주 급등 등 비정상 수익률 제거)
kr_rtn_raw  = kr_price.pct_change()
clip_mask_kr = (kr_rtn_raw < -0.9) | (kr_rtn_raw > 9.0)
n_clipped_kr = int(clip_mask_kr.sum().sum())
if n_clipped_kr > 0:
    top_kr = clip_mask_kr.sum(axis=0).nlargest(5)
    print(f"[KR 클리핑] 총 {n_clipped_kr}개 관측값 (-90%~+900% 범위 초과)")
    print(f"  클리핑 多 종목 Top5: {top_kr.to_dict()}")

kr_rtn = kr_rtn_raw.clip(-0.9, 9.0)
kr_rtn.columns = kr_rtn.columns.astype(str).str.zfill(6)
print(f'한국 수익률: {kr_rtn.shape[1]:,}종목 × {kr_rtn.shape[0]}개월')


## A-5. ME & B/M 팩터 구성 (한국)

In [ ]:
# ── A-5: ME & B/M 팩터 구성 (한국) ──────────────────────────────────────────
# ME (Market Equity, 시가총액):
#   ME_t = 발행주식수(Stocks) × 월말 주가(price_t)
#   Fama-French는 ME를 시가총액의 로그로 사용 → ln(ME)
#
# B/M (Book-to-Market ratio):
#   B/M_t = 자본총계(book equity) / 시가총액(ME_t)
#   Fama-French 래깅 관례:
#     - 7월 이후: 전년도(t-1년) 회계연도 장부가치 사용
#     - 7월 이전: 2년 전(t-2년) 회계연도 장부가치 사용
#   → 결산 후 재무제표 공시 지연(약 3~6개월)을 반영한 보수적 래깅
common_kr = list(set(kr_listing.index) & set(kr_price.columns))

price_kr  = kr_price[common_kr]
stocks_kr = pd.to_numeric(kr_listing.loc[common_kr, 'Stocks'], errors='coerce')

# ── ME 계산 ──────────────────────────────────────────────────────────────────
kr_ME    = price_kr.multiply(stocks_kr, axis=1)  # 주가 × 발행주식수 = 시가총액
kr_ME    = kr_ME[kr_ME > 0]                       # 음수·0 제거
kr_ln_ME = np.log(kr_ME)                          # 로그 변환 (정규성 개선)

print(f"ME 행렬: {kr_ln_ME.shape[0]}개월 × {kr_ln_ME.shape[1]:,}종목")
print(f"ln(ME) mean={kr_ln_ME.stack().mean():.2f} std={kr_ln_ME.stack().std():.2f}")

# ── B/M 계산 ─────────────────────────────────────────────────────────────────
# DART에서 수집한 book_df(자본총계)가 있는 경우에만 B/M 구성
if 'book_df' in globals() and not book_df.empty:
    bm_dict  = {}
    bm_codes = [t for t in common_kr if t in book_df.index]
    print(f"\nB/M 구성 대상: {len(bm_codes):,}개")

    for code in tqdm(bm_codes, desc='B/M', leave=False):
        be_s = book_df.loc[code].dropna()  # 해당 종목의 연도별 자본총계
        if be_s.empty:
            continue

        bm_vals = {}
        for date in kr_ME.index:
            if code not in kr_ME.columns:
                continue
            me = kr_ME.loc[date, code]
            if pd.isna(me) or me <= 0:
                continue

            # Fama-French 래깅: 7월 이후면 전년도, 이전이면 2년 전 회계연도
            ref_year = date.year - 1 if date.month >= 7 else date.year - 2
            be = be_s.get(ref_year, np.nan)  # 해당 회계연도의 자본총계
            if pd.isna(be) or be <= 0:
                continue  # 자본잠식 종목 제외

            bm = be / me  # B/M = 장부가치 / 시가총액
            if 0 < bm < 50:  # 극단적 B/M 필터링 (비정상 종목 제외)
                bm_vals[date] = bm

        if bm_vals:
            bm_dict[code] = pd.Series(bm_vals)

    kr_BM    = pd.DataFrame(bm_dict).reindex(kr_ME.index)
    kr_ln_BM = np.log(kr_BM[kr_BM > 0])
    have_bm  = kr_ln_BM.shape[1] > 0
    print(f"B/M 행렬: {kr_ln_BM.shape[0]}개월 × {kr_ln_BM.shape[1]:,}종목")

else:
    # DART 데이터 없는 경우 Beta + ln(ME) 두 팩터만으로 FM 진행
    have_bm  = False
    kr_ln_BM = pd.DataFrame()
    print("\nbook_df 없음 → Beta + ln(ME) 만으로 FM 진행")
    print("(DART API 키 정상화 후 B/M 추가 가능)")

print(f"\nhave_bm = {have_bm}")

# ── 컬럼 타입 통일 (종목코드를 6자리 문자열로 표준화) ───────────────────────
# 이후 셀에서 set 교집합 연산 시 타입 불일치로 교집합이 0이 되는 문제 방지
kr_rtn.columns   = kr_rtn.columns.astype(str).str.zfill(6)
kr_ln_ME.columns = kr_ln_ME.columns.astype(str).str.zfill(6)
if have_bm:
    kr_ln_BM.columns = kr_ln_BM.columns.astype(str).str.zfill(6)


## A-6. CAPM 롤링 베타 (한국)

In [ ]:
# ── A-6: CAPM 롤링 베타 추정 (한국) ──────────────────────────────────────────
# 베타를 추정할 대상 = 가격 데이터(kr_rtn) AND ME 데이터(kr_ln_ME) 모두 있는 종목.
# rolling_beta 함수는 완전 벡터화로 구현돼 있어 2,000+ 종목도 빠르게 처리.
# 결과: kr_beta (T행 × N열) — 각 셀은 해당 시점의 60개월 롤링 베타 추정치.
beta_tickers_kr = list(set(kr_rtn.columns) & set(kr_ln_ME.columns))
kr_rtn_beta     = kr_rtn[beta_tickers_kr]

print(f'베타 추정 대상: {len(beta_tickers_kr):,}종목')
kr_beta = rolling_beta(kr_rtn_beta, kr_mkt_rtn)

# 베타 분포 요약 (정상 범위: 0~2; 극단값 있으면 데이터 품질 점검)
bv = kr_beta.stack().dropna()
print(f'beta | mean={bv.mean():.3f} std={bv.std():.3f} ',
      f'p1={bv.quantile(0.01):.2f} p99={bv.quantile(0.99):.2f}')


## A-7. Fama-MacBeth 회귀 — 한국

In [ ]:
# ── A-7: Fama-MacBeth 회귀 — 한국 ───────────────────────────────────────────
# run_fama_macbeth()는 Model 1~3을 순차 실행:
#   Model 1: Beta only            → CAPM의 단독 설명력
#   Model 2: Beta + ln(ME) + ln(B/M)  → FF(1992) 완전 모형
#   Model 3: ln(ME) + ln(B/M) only   → 베타 없이 사이즈·가치 효과만
#
# 반환값:
#   kr_res_m2      : Model 2 결과 (팩터별 평균 gamma, t-stat, 유의성)
#   kr_gamma_m2    : Model 2 월별 gamma 시계열 (C-2 시각화에 사용)
#   common_all_kr  : 3개 팩터 공통 종목 리스트
kr_res_m1, kr_res_m2, kr_res_m3, kr_gamma_m2, common_all_kr = run_fama_macbeth(
    kr_rtn, kr_beta, kr_ln_ME, kr_ln_BM, label='한국'
)

# 이후 C-3 포트폴리오 분석 셀에서 사용하는 변수 정의
rtn_kr_fm   = kr_rtn[common_all_kr]
kr_gamma_m1 = None  # Model 1 gamma가 필요하면 run_fama_macbeth 수정
factor_dict_kr = {
    'Beta':  kr_beta[common_all_kr],
    'ln_ME': kr_ln_ME[common_all_kr],
    'ln_BM': kr_ln_BM[common_all_kr],
}


---
## B. 미국

## B-5. ME & B/M 팩터 구성 (미국)

In [ ]:
# ── B-5: ME & B/M 팩터 구성 (미국) ──────────────────────────────────────────
# 한국(Cell 22)과 동일한 방법론으로 미국 팩터를 구성:
#   ME_t = 주식수(shares) × 월말 주가(price_t)
#   B/M_t = Stockholders Equity(래깅) / ME_t
# Fama-French 래깅 규칙: 7월 이후 → 전년도, 7월 이전 → 2년 전 회계연도
common_us = [t for t in valid_us.index if t in us_price.columns]

price_us  = us_price[common_us]
shares_us = valid_us.loc[common_us, 'shares']

# ── ME 계산 ──────────────────────────────────────────────────────────────────
us_ME    = price_us.multiply(shares_us, axis=1)  # 주가 × 주식수 = 시가총액 (USD)
us_ME    = us_ME[us_ME > 0]
us_ln_ME = np.log(us_ME)

# ── B/M 계산 (연도별 장부가치 래깅 적용) ─────────────────────────────────────
bm_us_dict = {}
for ticker in tqdm(common_us, desc='US B/M', leave=False):
    if ticker not in us_bv_annual or not us_bv_annual[ticker]:
        continue  # 장부가치 없는 종목 스킵

    be_s = pd.Series(us_bv_annual[ticker]).dropna()  # {year: equity}
    if be_s.empty:
        continue

    bm_vals = {}
    for date in us_ME.index:
        me = us_ME.loc[date, ticker] if ticker in us_ME.columns else np.nan
        if pd.isna(me) or me <= 0:
            continue

        # Fama-French 래깅: 7월 이후면 전년도 재무제표, 이전이면 2년 전
        # (결산 후 공시까지 최소 6개월 소요를 보수적으로 반영)
        ref_year = date.year - 1 if date.month >= 7 else date.year - 2
        be = be_s.get(ref_year, np.nan)
        if pd.isna(be) or be <= 0:
            continue  # 자본잠식 또는 데이터 없는 경우 제외

        bm = be / me  # B/M = 장부가치 / 시가총액
        if 0 < bm < 50:  # 극단적 B/M 필터 (비정상 종목 제외)
            bm_vals[date] = bm

    if bm_vals:
        bm_us_dict[ticker] = pd.Series(bm_vals)

us_BM    = pd.DataFrame(bm_us_dict).reindex(us_ME.index)
us_BM    = us_BM[(us_BM > 0) & (us_BM < 50)]
us_ln_BM = np.log(us_BM)

print(f'미국 ME  행렬: {us_ln_ME.shape[0]}개월 x {us_ln_ME.shape[1]:,}종목')
print(f'미국 B/M 행렬: {us_ln_BM.shape[0]}개월 x {us_ln_BM.shape[1]:,}종목')
print(f'ln(ME)  분포: mean={us_ln_ME.stack().mean():.2f}, std={us_ln_ME.stack().std():.2f}')
print(f'ln(B/M) 분포: mean={us_ln_BM.stack().mean():.3f}, std={us_ln_BM.stack().std():.3f}')


## B-6. CAPM 롤링 베타 (미국)

In [ ]:
# ── B-6: CAPM 롤링 베타 추정 (미국) ──────────────────────────────────────────
# 한국(Cell 25)과 동일한 rolling_beta 함수 사용.
# 기준 시장 포트폴리오: S&P 500(us_mkt_rtn)
# 대상: 수익률(us_rtn) AND ME(us_ln_ME) 모두 있는 종목
beta_tickers_us = list(set(us_rtn.columns) & set(us_ln_ME.columns))
us_rtn_beta     = us_rtn[beta_tickers_us]

print(f'베타 추정 대상: {len(beta_tickers_us)}종목')
us_beta = rolling_beta(us_rtn_beta, us_mkt_rtn)

bv_us = us_beta.stack().dropna()
print(f'beta | mean={bv_us.mean():.3f} std={bv_us.std():.3f}',
      f'p1={bv_us.quantile(0.01):.2f} p99={bv_us.quantile(0.99):.2f}')


## B-7. Fama-MacBeth 회귀 — 미국

In [ ]:
# ── B-7: Fama-MacBeth 회귀 — 미국 ───────────────────────────────────────────
# 한국(Cell 28)과 동일한 run_fama_macbeth() 함수를 사용해 3개 모델 실행.
# 결과를 한국과 나란히 비교하는 것이 PART C의 목적.
us_res_m1, us_res_m2, us_res_m3, us_gamma_m2, common_all_us = run_fama_macbeth(
    us_rtn, us_beta, us_ln_ME, us_ln_BM, label='미국'
)

# 이후 C-3 포트폴리오 분석 셀에서 사용하는 변수 정의
rtn_us_fm = us_rtn[common_all_us]
factor_dict_us = {
    'Beta':  us_beta[common_all_us],
    'ln_ME': us_ln_ME[common_all_us],
    'ln_BM': us_ln_BM[common_all_us],
}


---

## C-1. 결과 요약 테이블

In [ ]:
# ── C-1: 한국 vs 미국 결과 요약 테이블 ──────────────────────────────────────
# Model 2(Beta + ln(ME) + ln(B/M)) 기준으로 두 시장 결과를 나란히 비교.
# 해석 기준:
#   gamma_1(Beta) 비유의 → CAPM 실패 (FF 1992 핵심 주장)
#   gamma_2(ln_ME) 음수+유의 → 소형주 프리미엄 존재
#   gamma_3(ln_BM) 양수+유의 → 가치주 프리미엄 존재

def fmt_result(res_df, label):
    """fm_step2 결과를 비교용 테이블 행으로 변환 (상수항 제외)."""
    rows = []
    for factor in [c for c in res_df.index if c != 'const']:
        row = res_df.loc[factor]
        rows.append({
            'Market':     label,
            'Factor':     factor,
            'Mean gamma': f"{row['Mean_gamma']:+.4f}",
            't-stat':     f"{row['t_stat']:+.3f}{row['Sig']}",
            'N months':   int(row['N_months']),
        })
    return pd.DataFrame(rows)

rows_kr    = fmt_result(kr_res_m2, 'Korea')
rows_us    = fmt_result(us_res_m2, 'USA')
compare_df = pd.concat([rows_kr, rows_us]).set_index(['Market', 'Factor'])

print('=' * 70)
print('  Fama-MacBeth Model 2: Beta + ln(ME) + ln(B/M)')
print(f'  기간: {DATE_START} ~ {DATE_END}')
print(f'  표준오차: Newey-West HAC (lag={NW_LAGS}개월)')
print('-' * 70)
print(compare_df.to_string())
print('=' * 70)
print('* p<10%  ** p<5%  *** p<1%')
print()
print('[해석 기준]')
print('  Beta(gamma_1): 비유의 → CAPM 실패 (FF1992 주장)')
print('  ln_ME(gamma_2): 음수+유의 → 소형주 프리미엄')
print('  ln_BM(gamma_3): 양수+유의 → 가치주 프리미엄')


## C-2. gamma 시계열 시각화

In [ ]:
# ── C-2: gamma 시계열 + 누적합 시각화 ───────────────────────────────────────
# 왼쪽: 월별 gamma_t 시계열 (12개월 이동평균 오버레이)
#   → 팩터 프리미엄의 시간적 안정성 확인
# 오른쪽: gamma_t 누적합
#   → 장기적으로 한 방향으로 누적되면 안정적 프리미엄 존재를 시사
factors = ['Beta', 'ln_ME', 'ln_BM']
factor_labels = {
    'Beta':  'gamma_1 (Beta)',
    'ln_ME': 'gamma_2 (ln ME)  [소형주: 음수]',
    'ln_BM': 'gamma_3 (ln B/M) [가치주: 양수]',
}
colors = {'Korea': 'steelblue', 'USA': 'firebrick'}

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle('Fama-MacBeth gamma 시계열: 한국(KOSPI+KOSDAQ) vs 미국(S&P500)',
             fontsize=14, fontweight='bold')

market_data = {
    'Korea': {'gamma': kr_gamma_m2, 'result': kr_res_m2},
    'USA':   {'gamma': us_gamma_m2, 'result': us_res_m2},
}

for row_i, fac in enumerate(factors):
    ax_ts  = axes[row_i, 0]  # 왼쪽: 시계열
    ax_cum = axes[row_i, 1]  # 오른쪽: 누적합

    for mkt, d in market_data.items():
        g   = d['gamma']
        r   = d['result']
        col = colors[mkt]

        if fac not in g.columns:
            continue

        series = g[fac].dropna()

        # 반투명 원시 시계열 + 12개월 이동평균 (추세 파악)
        ax_ts.plot(series.index, series.values, alpha=0.3, lw=0.8, color=col)
        ax_ts.plot(series.rolling(12).mean().dropna(), lw=2, color=col,
                   label=f'{mkt}: gamma={r.loc[fac,"Mean_gamma"]:+.4f} '
                         f'(t={r.loc[fac,"t_stat"]:+.2f}{r.loc[fac,"Sig"]})')

        # 누적합: 양(+) 방향으로 우상향이면 안정적 프리미엄
        ax_cum.plot(series.cumsum(), lw=2, color=col, label=mkt)

    ax_ts.axhline(0, color='black', lw=1)
    ax_ts.set_title(factor_labels[fac])
    ax_ts.legend(fontsize=8)
    ax_ts.set_ylabel('gamma')

    ax_cum.axhline(0, color='black', lw=1)
    ax_cum.set_title(f'{factor_labels[fac]} (누적합)')
    ax_cum.legend(fontsize=9)
    ax_cum.set_ylabel('Cumulative gamma')

plt.tight_layout()
plt.savefig('fm_korea_vs_usa.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: fm_korea_vs_usa.png')


## C-3. Beta & Size 10분위 포트폴리오

In [ ]:
# ── C-3: Beta & Size 10분위 포트폴리오 수익률 비교 ──────────────────────────
# Fama-MacBeth 회귀 결과를 보완하는 비모수적 검증 방법.
# 팩터값(베타 or 사이즈)을 10분위로 나눠 각 분위의 평균 수익률을 비교:
#   - Beta 10분위: 저베타(1)→고베타(10) 수익률이 단조증가하면 CAPM 지지,
#                 단조증가 없으면 CAPM 실패를 의미 (FF 1992 핵심 발견)
#   - Size 10분위: 소형주(1)→대형주(10) 수익률이 단조감소하면 소형주 프리미엄 지지

def decile_portfolio(rtn_df, factor_df, desc=''):
    """팩터값 기준 10분위 포트폴리오 월평균 수익률과 표준오차를 계산.

    매월 t:
      1) 팩터값을 t-1 시점으로 래깅 (look-ahead bias 방지)
      2) 10분위로 분류 → 각 분위별 동일가중 수익률 계산
    최종: 월별 수익률의 시계열 평균 + 표준오차(= std/sqrt(T))

    Parameters
    ----------
    rtn_df    : 종목별 월별 수익률 (T×N)
    factor_df : 팩터값 (T×N), 래깅 전 값
    desc      : 진행 표시용 설명 문자열

    Returns
    -------
    mean : {분위: 월평균 수익률(%)}
    se   : {분위: 표준오차(%)}
    """
    factor_lag   = factor_df.shift(1)  # 1개월 래깅
    common_dates = factor_lag.index.intersection(rtn_df.index)
    port = {i: [] for i in range(1, 11)}  # 10개 분위 버킷

    for date in common_dates:
        f_t = factor_lag.loc[date].dropna()
        r_t = rtn_df.loc[date].dropna()
        common = f_t.index.intersection(r_t.index)
        if len(common) < 50:
            continue  # 종목 수 부족 → 안정적 분위 구성 불가

        # pd.qcut: 동일 개수 기준 10분위 (1=최솟값, 10=최댓값)
        deciles = pd.qcut(f_t.loc[common], 10, labels=False, duplicates='drop') + 1
        for d in deciles.dropna().unique().astype(int):
            # 해당 분위 종목들의 동일가중 평균 수익률
            port[d].append(r_t.loc[deciles[deciles == d].index].mean())

    # 시계열 평균 + 표준오차 (단위: %)
    mean = {d: np.mean(v) * 100 for d, v in port.items() if v}
    se   = {d: np.std(v) * 100 / np.sqrt(len(v)) for d, v in port.items() if v}
    return mean, se


print('Beta 10분위 계산...')
kr_beta_port_mean, kr_beta_port_se = decile_portfolio(kr_rtn, kr_beta, 'KR Beta')
us_beta_port_mean, us_beta_port_se = decile_portfolio(us_rtn, us_beta, 'US Beta')

print('Size 10분위 계산...')
kr_size_port_mean, kr_size_port_se = decile_portfolio(kr_rtn, kr_ln_ME, 'KR Size')
us_size_port_mean, us_size_port_se = decile_portfolio(us_rtn, us_ln_ME, 'US Size')

# ── 시각화 (2×2 서브플롯) ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('10분위 포트폴리오 월평균 수익률: 한국 vs 미국', fontsize=13, fontweight='bold')

plot_data = [
    (axes[0, 0], kr_beta_port_mean, kr_beta_port_se, 'steelblue',
     'Beta 10분위 — 한국', 'Beta (1=저베타, 10=고베타)'),
    (axes[0, 1], us_beta_port_mean, us_beta_port_se, 'firebrick',
     'Beta 10분위 — 미국', 'Beta (1=저베타, 10=고베타)'),
    (axes[1, 0], kr_size_port_mean, kr_size_port_se, 'steelblue',
     'Size 10분위 — 한국', 'Size (1=소형주, 10=대형주)'),
    (axes[1, 1], us_size_port_mean, us_size_port_se, 'firebrick',
     'Size 10분위 — 미국', 'Size (1=소형주, 10=대형주)'),
]

for ax, mean_d, se_d, color, title, xlabel in plot_data:
    xs = list(mean_d.keys())
    ys = list(mean_d.values())
    es = list(se_d.values())
    # 오차 막대(±1 표준오차) 포함 막대그래프
    ax.bar(xs, ys, yerr=es, color=color, alpha=0.8, edgecolor='white', capsize=4)
    ax.axhline(0, color='black', lw=1)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('월평균 수익률 (%)')
    ax.set_xticks(xs)

plt.tight_layout()
plt.savefig('fm_decile_compare.png', dpi=150, bbox_inches='tight')
plt.show()


## C-4. 결과 해석 요약



### 모델별 판단 기준



| 팩터 | 방향 | FF(1992) 미국 원 논문 | 한국 시장 예상 | 본 분석 결과 |

|------|------|---------------------|-------------|------------|

| Beta (gamma_1) | - | 비유의 (t~0) | 비유의 혹은 약 음수 | 결과 확인 |

| ln_ME (gamma_2) | 음수 | 유의 (t≈-2.58) | 유의 (소형주 프리미엄) | 결과 확인 |

| ln_BM (gamma_3) | 양수 | 유의 (t≈+2.57) | 한국은 약할 수 있음 | 결과 확인 |



### B/M 팩터 구성 방법

- 한국: DART `fnlttMultiAcnt`로 연도별 자본총계 수집 → Fama-French 래깅 적용

- 미국: yfinance `balance_sheet`에서 연도별 Stockholders Equity 수집 → 동일 래깅 적용

- **한계**: yfinance는 최근 4개 회계연도만 제공 → 2015~2018년 구간 B/M 결측 가능성 있음

### ⚠️ 생존편향 주의

- 현재 KOSPI+KOSDAQ / S&P500 구성종목만 사용 → 상장폐지 종목 제외

- 팩터 프리미엄(소형주·가치주)이 실제보다 **과대추정**될 수 있음

- 정확한 분석: CRSP(미국) + KRX 전종목 이력 데이터 사용 권장



### 한국 vs 미국 해석 포인트

1. **Beta 계수 비교**: 어느 시장에서 CAPM이 더 강하게 실패하는가?

2. **Size 효과 크기**: `|gamma_2|`의 크기 — 한국 소형주 프리미엄이 더 클 수 있음

3. **Value 효과**: 한국은 B/M 효과가 미국보다 약하거나 역전될 수 있음 (성장주 장세)

4. **t-stat**: 미국은 월 500종목+, 한국은 2000종목+ → 한국이 더 많은 횡단면 정보